# Spatial Analysis with GeoPandas

This notebook demonstrates common spatial analysis operations on GIS data.

**What you'll learn:**
- Spatial queries (intersection, contains, within)
- Buffer operations
- Distance calculations
- Spatial joins
- Area and length calculations
- Coordinate reference system transformations

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for inline display
%matplotlib inline

## 1. Load Data

In [ ]:
# Configure geodatabase path
GDB_PATH = "../data-extracted/RVO_NW_I_GGM.gdb"

def load_layer(gdb_path, layer_name, target_crs="EPSG:4326"):
    """
    Load a layer and transform to target CRS.
    """
    try:
        gdf = gpd.read_file(gdb_path, layer=layer_name)
        gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty]
        if gdf.crs and gdf.crs != target_crs:
            gdf = gdf.to_crs(target_crs)
        return gdf
    except Exception as e:
        print(f"Error loading {layer_name}: {e}")
        return None

In [ ]:
import fiona

# List available layers
layers = fiona.listlayers(GDB_PATH)
print(f"Available layers: {len(layers)}")

# Find layers by geometry type
point_layers = []
polygon_layers = []
line_layers = []

for layer_name in layers:
    try:
        with fiona.open(GDB_PATH, layer=layer_name) as src:
            geom_type = src.schema.get('geometry')
            if geom_type and len(src) > 0:
                if 'Point' in str(geom_type):
                    point_layers.append(layer_name)
                elif 'Polygon' in str(geom_type):
                    polygon_layers.append(layer_name)
                elif 'Line' in str(geom_type):
                    line_layers.append(layer_name)
    except:
        pass

print(f"\nPoint layers: {len(point_layers)}")
print(f"Polygon layers: {len(polygon_layers)}")
print(f"Line layers: {len(line_layers)}")

In [ ]:
# Load sample layers for analysis
# Modify these to match your data

sample_polygon = None
sample_points = None
sample_lines = None

if polygon_layers:
    sample_polygon = load_layer(GDB_PATH, polygon_layers[0])
    print(f"Loaded polygon layer: {polygon_layers[0]} ({len(sample_polygon)} features)")

if point_layers:
    sample_points = load_layer(GDB_PATH, point_layers[0])
    print(f"Loaded point layer: {point_layers[0]} ({len(sample_points)} features)")

if line_layers:
    sample_lines = load_layer(GDB_PATH, line_layers[0])
    print(f"Loaded line layer: {line_layers[0]} ({len(sample_lines)} features)")

## 2. Coordinate Reference Systems

In [ ]:
# Understanding CRS
if sample_polygon is not None:
    print(f"Current CRS: {sample_polygon.crs}")
    print(f"CRS units: {sample_polygon.crs.axis_info[0].unit_name if sample_polygon.crs else 'Unknown'}")

In [ ]:
# Transform to a projected CRS for accurate distance/area calculations
# UTM Zone 31N is suitable for the North Sea area
PROJECTED_CRS = "EPSG:32631"  # UTM Zone 31N

def to_projected(gdf):
    """Transform GeoDataFrame to projected CRS for metric calculations."""
    if gdf is None:
        return None
    return gdf.to_crs(PROJECTED_CRS)

# Create projected versions
polygon_proj = to_projected(sample_polygon)
points_proj = to_projected(sample_points)
lines_proj = to_projected(sample_lines)

## 3. Area and Length Calculations

In [ ]:
# Calculate areas for polygons (in square meters)
if polygon_proj is not None and len(polygon_proj) > 0:
    polygon_proj['area_m2'] = polygon_proj.geometry.area
    polygon_proj['area_km2'] = polygon_proj['area_m2'] / 1_000_000
    
    print("Area Statistics (km2):")
    print(f"  Total: {polygon_proj['area_km2'].sum():.2f}")
    print(f"  Mean: {polygon_proj['area_km2'].mean():.4f}")
    print(f"  Min: {polygon_proj['area_km2'].min():.4f}")
    print(f"  Max: {polygon_proj['area_km2'].max():.4f}")

In [ ]:
# Calculate lengths for lines (in meters)
if lines_proj is not None and len(lines_proj) > 0:
    lines_proj['length_m'] = lines_proj.geometry.length
    lines_proj['length_km'] = lines_proj['length_m'] / 1000
    
    print("Length Statistics (km):")
    print(f"  Total: {lines_proj['length_km'].sum():.2f}")
    print(f"  Mean: {lines_proj['length_km'].mean():.4f}")
    print(f"  Min: {lines_proj['length_km'].min():.4f}")
    print(f"  Max: {lines_proj['length_km'].max():.4f}")

## 4. Buffer Operations

In [ ]:
# Create buffers around points (in meters, using projected CRS)
if points_proj is not None and len(points_proj) > 0:
    buffer_distance = 500  # meters
    
    # Create buffer
    points_buffered = points_proj.copy()
    points_buffered['geometry'] = points_proj.geometry.buffer(buffer_distance)
    
    print(f"Created {buffer_distance}m buffers around {len(points_buffered)} points")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    points_buffered.plot(ax=ax, alpha=0.3, color='blue', edgecolor='blue')
    points_proj.plot(ax=ax, color='red', markersize=10)
    ax.set_title(f'{buffer_distance}m Buffers Around Points')
    plt.show()

## 5. Distance Calculations

In [ ]:
# Calculate distances between points
if points_proj is not None and len(points_proj) >= 2:
    # Distance from first point to all others
    reference_point = points_proj.geometry.iloc[0]
    points_proj['distance_to_first'] = points_proj.geometry.distance(reference_point)
    
    print(f"Distances from first point (m):")
    print(points_proj['distance_to_first'].describe())

In [ ]:
# Find nearest neighbor for each point
def find_nearest(gdf):
    """Find nearest neighbor distance for each feature."""
    distances = []
    for idx, row in gdf.iterrows():
        other_points = gdf[gdf.index != idx]
        if len(other_points) > 0:
            min_dist = other_points.geometry.distance(row.geometry).min()
            distances.append(min_dist)
        else:
            distances.append(np.nan)
    return distances

if points_proj is not None and len(points_proj) > 1:
    points_proj['nearest_neighbor_dist'] = find_nearest(points_proj)
    
    print("Nearest Neighbor Distances (m):")
    print(f"  Mean: {points_proj['nearest_neighbor_dist'].mean():.2f}")
    print(f"  Min: {points_proj['nearest_neighbor_dist'].min():.2f}")
    print(f"  Max: {points_proj['nearest_neighbor_dist'].max():.2f}")

## 6. Spatial Queries

In [ ]:
# Create a bounding box for spatial queries
if sample_polygon is not None and len(sample_polygon) > 0:
    bounds = sample_polygon.total_bounds  # [minx, miny, maxx, maxy]
    center_x = (bounds[0] + bounds[2]) / 2
    center_y = (bounds[1] + bounds[3]) / 2
    
    # Create a query box (smaller than total extent)
    query_box = box(
        center_x - 0.1,  # degrees
        center_y - 0.1,
        center_x + 0.1,
        center_y + 0.1
    )
    query_gdf = gpd.GeoDataFrame(geometry=[query_box], crs="EPSG:4326")
    
    print(f"Query box created at center: {center_x:.4f}, {center_y:.4f}")

In [ ]:
# Spatial query: find features within the box
if sample_points is not None and 'query_box' in dir():
    # Points within box
    points_in_box = sample_points[sample_points.geometry.within(query_box)]
    print(f"Points within query box: {len(points_in_box)} of {len(sample_points)}")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    query_gdf.boundary.plot(ax=ax, color='red', linewidth=2)
    sample_points.plot(ax=ax, color='blue', markersize=5, alpha=0.5, label='All points')
    points_in_box.plot(ax=ax, color='green', markersize=20, label='Points in box')
    ax.legend()
    ax.set_title('Spatial Query: Points Within Box')
    plt.show()

## 7. Spatial Joins

In [ ]:
# Spatial join: points within polygons
if sample_points is not None and sample_polygon is not None:
    if len(sample_points) > 0 and len(sample_polygon) > 0:
        # Join points to polygons
        joined = gpd.sjoin(
            sample_points, 
            sample_polygon, 
            how='inner', 
            predicate='within'
        )
        
        print(f"Spatial join results:")
        print(f"  Original points: {len(sample_points)}")
        print(f"  Points within polygons: {len(joined)}")
        
        if len(joined) > 0:
            print(f"\nJoined columns: {list(joined.columns[:10])}")

In [ ]:
# Count points per polygon
if sample_points is not None and sample_polygon is not None:
    if len(sample_points) > 0 and len(sample_polygon) > 0:
        # Add a unique ID to polygons if not present
        polygon_copy = sample_polygon.copy()
        polygon_copy['polygon_id'] = range(len(polygon_copy))
        
        # Spatial join
        joined = gpd.sjoin(sample_points, polygon_copy[['polygon_id', 'geometry']], predicate='within')
        
        # Count per polygon
        if len(joined) > 0:
            counts = joined.groupby('polygon_id').size()
            print(f"Points per polygon:")
            print(counts.describe())

## 8. Overlay Operations

In [ ]:
# Intersection of two polygon layers
# This requires two polygon layers with overlapping features

if polygon_layers and len(polygon_layers) >= 2:
    layer1 = load_layer(GDB_PATH, polygon_layers[0])
    layer2 = load_layer(GDB_PATH, polygon_layers[1])
    
    if layer1 is not None and layer2 is not None:
        if len(layer1) > 0 and len(layer2) > 0:
            try:
                intersection = gpd.overlay(layer1, layer2, how='intersection')
                print(f"Intersection result: {len(intersection)} features")
            except Exception as e:
                print(f"Intersection failed: {e}")

## 9. Export Results

In [ ]:
# Export analysis results to GeoJSON
if points_proj is not None and len(points_proj) > 0:
    # Convert back to WGS84 for export
    export_gdf = points_proj.to_crs("EPSG:4326")
    
    # Save to file
    output_file = 'analysis_results.geojson'
    export_gdf.to_file(output_file, driver='GeoJSON')
    print(f"Results saved to: {output_file}")

In [ ]:
# Export statistics to CSV
if points_proj is not None and len(points_proj) > 0:
    # Create statistics summary
    stats = {
        'total_features': len(points_proj),
        'crs': str(points_proj.crs),
    }
    
    # Add column statistics
    for col in points_proj.select_dtypes(include=[np.number]).columns:
        if col != 'geometry':
            stats[f'{col}_mean'] = points_proj[col].mean()
            stats[f'{col}_sum'] = points_proj[col].sum()
    
    stats_df = pd.DataFrame([stats])
    stats_df.to_csv('analysis_statistics.csv', index=False)
    print("Statistics saved to: analysis_statistics.csv")

## Summary

This notebook demonstrated:

1. **CRS Transformations** - Converting between geographic and projected coordinate systems
2. **Area/Length Calculations** - Computing geometric measurements in proper units
3. **Buffer Operations** - Creating zones around features
4. **Distance Calculations** - Measuring distances between features
5. **Spatial Queries** - Finding features within specific areas
6. **Spatial Joins** - Combining data based on spatial relationships
7. **Overlay Operations** - Computing geometric intersections

For more information, see the [GeoPandas documentation](https://geopandas.org/).